In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict

In [12]:
import pandas as pd

data = pd.read_csv(
    r"C:\Users\Karishma\Downloads\NER dataset.csv\NER dataset.csv",
    encoding="latin1"
)

print(data.head())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


In [15]:
print(data.shape)

print(data.info())

print(data.isnull().sum())

(1048575, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   Sentence #  47959 non-null    object
 1   Word        1048565 non-null  object
 2   POS         1048575 non-null  object
 3   Tag         1048575 non-null  object
dtypes: object(4)
memory usage: 32.0+ MB
None
Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64


In [16]:
data["Sentence #"] = data["Sentence #"].ffill()

print(data.head())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1  Sentence: 1             of   IN   O
2  Sentence: 1  demonstrators  NNS   O
3  Sentence: 1           have  VBP   O
4  Sentence: 1        marched  VBN   O


In [17]:
sentences = data.groupby("Sentence #")["Word"].apply(list).tolist()

tags = data.groupby("Sentence #")["POS"].apply(list).tolist()

print("Total Sentences:", len(sentences))

print(sentences[0])

print(tags[0])

Total Sentences: 47959
['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']
['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP', 'TO', 'VB', 'DT', 'NN', 'IN', 'NNP', 'CC', 'VB', 'DT', 'NN', 'IN', 'JJ', 'NNS', 'IN', 'DT', 'NN', '.']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    sentences,
    tags,
    test_size=0.2,
    random_state=42
)

print("Training:", len(X_train))
print("Testing:", len(X_test))

Training: 38367
Testing: 9592


In [19]:
vocab = set()

for sentence in X_train:
    vocab.update(sentence)

vocab = list(vocab)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 31917


In [20]:
tagset = set()

for tag_sequence in y_train:
    tagset.update(tag_sequence)

tagset = list(tagset)

print(tagset)

print("Total POS Tags:", len(tagset))

['MD', 'POS', 'VBG', ';', 'NN', 'WP', 'RRB', 'DT', 'NNS', 'WDT', 'CC', 'VBN', ':', 'UH', 'FW', 'RBS', ',', 'CD', 'RBR', 'PDT', 'JJS', 'WP$', 'NNP', 'WRB', 'VB', 'VBD', 'RP', 'NNPS', '$', 'PRP', 'TO', 'EX', 'RB', '.', 'VBZ', '``', 'LRB', 'PRP$', 'JJR', 'VBP', 'IN', 'JJ']
Total POS Tags: 42


In [21]:
word2idx = {w: i for i, w in enumerate(vocab)}

tag2idx = {t: i for i, t in enumerate(tagset)}

idx2tag = {i: t for t, i in tag2idx.items()}

In [22]:
num_tags = len(tagset)

initial = np.ones(num_tags)

for tag_sequence in y_train:
    initial[tag2idx[tag_sequence[0]]] += 1

initial = initial / initial.sum()

print(initial)

[2.60355646e-05 2.60355646e-05 9.86747898e-03 2.60355646e-05
 2.23124788e-02 1.82248952e-04 2.60355646e-05 3.06568773e-01
 8.07883569e-02 7.81066937e-05 1.99172069e-02 2.81184097e-03
 1.04142258e-04 5.20711292e-05 2.60355646e-05 2.60355646e-05
 2.60355646e-05 1.41112760e-02 4.73847275e-03 7.81066937e-05
 1.82248952e-03 2.60355646e-05 2.45775730e-01 1.40592049e-03
 1.82248952e-04 2.60355646e-05 2.60355646e-05 6.24853550e-04
 2.60355646e-05 6.76403968e-02 7.55031373e-04 4.71243719e-03
 3.15551043e-02 2.60355646e-05 2.60355646e-05 2.42130751e-03
 5.20711292e-05 6.04025098e-03 4.55622380e-03 5.20711292e-05
 7.78203025e-02 9.26345388e-02]


In [23]:
transition = np.ones((num_tags, num_tags))

for tag_sequence in y_train:
    for i in range(len(tag_sequence) - 1):
        current_tag = tag2idx[tag_sequence[i]]
        next_tag = tag2idx[tag_sequence[i + 1]]
        transition[current_tag][next_tag] += 1

transition = transition / transition.sum(axis=1, keepdims=True)

print("Transition Matrix Shape:", transition.shape)

Transition Matrix Shape: (42, 42)


In [24]:
num_words = len(vocab)

emission = np.ones((num_tags, num_words))

for words, tag_sequence in zip(X_train, y_train):
    for word, tag in zip(words, tag_sequence):
        emission[tag2idx[tag]][word2idx[word]] += 1

emission = emission / emission.sum(axis=1, keepdims=True)

print("Emission Matrix Shape:", emission.shape)

Emission Matrix Shape: (42, 31917)


In [25]:
log_initial = np.log(initial)
log_transition = np.log(transition)
log_emission = np.log(emission)

print("Log probabilities created successfully.")

Log probabilities created successfully.


In [30]:
def viterbi(sentence):
    T = len(sentence)
    N = len(tagset)

    dp = np.full((N, T), -np.inf)
    backpointer = np.zeros((N, T), dtype=int)

    if sentence[0] in word2idx:
        emission_prob = log_emission[:, word2idx[sentence[0]]]
    else:
        emission_prob = np.log(np.ones(N) / N)

    dp[:, 0] = log_initial + emission_prob

    for t in range(1, T):

        if sentence[t] in word2idx:
            emission_prob = log_emission[:, word2idx[sentence[t]]]
        else:
            emission_prob = np.log(np.ones(N) / N)

        scores = dp[:, t-1][:, None] + log_transition

        backpointer[:, t] = np.argmax(scores, axis=0)

        dp[:, t] = np.max(scores, axis=0) + emission_prob

    best = np.zeros(T, dtype=int)
    best[-1] = np.argmax(dp[:, T-1])

    for t in range(T-2, -1, -1):
        best[t] = backpointer[best[t+1], t+1]

    return [idx2tag[i] for i in best]

In [31]:
predicted = []
actual = []

for sentence, true_tags in zip(X_test, y_test):
    pred = viterbi(sentence)
    predicted.extend(pred)
    actual.extend(true_tags)

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [34]:
print("Accuracy:", accuracy_score(actual, predicted))
print("Precision:", precision_score(actual, predicted, average="weighted"))
print("Recall:", recall_score(actual, predicted, average="weighted"))
print("F1 Score:", f1_score(actual, predicted, average="weighted"))

print("\nClassification Report:")
print(classification_report(actual, predicted))

Accuracy: 0.9447457465567364
Precision: 0.9454165766573359
Recall: 0.9447457465567364
F1 Score: 0.9440872285149948

Classification Report:
              precision    recall  f1-score   support

           $       0.98      1.00      0.99       208
           ,       0.97      1.00      0.99      6544
           .       0.95      1.00      0.98      9565
           :       1.00      0.31      0.47       167
           ;       0.95      0.44      0.60        43
          CC       1.00      1.00      1.00      4618
          CD       0.97      0.92      0.95      4983
          DT       0.95      1.00      0.97     19735
          EX       1.00      0.78      0.88       137
          IN       0.96      0.99      0.98     24247
          JJ       0.89      0.91      0.90     15524
         JJR       0.86      0.85      0.86       557
         JJS       0.94      0.87      0.90       638
         LRB       1.00      0.89      0.94       137
          MD       0.97      0.99      0.98      1

In [35]:
test_sentences = [
    ["I", "love", "Python"],
    ["She", "is", "reading", "a", "book"],
    ["Dogs", "are", "playing", "outside"],
    ["The", "weather", "is", "beautiful"],
    ["Artificial", "Intelligence", "changes", "the", "world"]
]

for sentence in test_sentences:
    predicted_tags = viterbi(sentence)

    print("Sentence :", " ".join(sentence))
    print("POS Tags :", predicted_tags)
    print("-" * 50)

Sentence : I love Python
POS Tags : ['PRP', 'MD', 'VB']
--------------------------------------------------
Sentence : She is reading a book
POS Tags : ['PRP', 'VBZ', 'VBG', 'DT', 'NN']
--------------------------------------------------
Sentence : Dogs are playing outside
POS Tags : ['NNS', 'VBP', 'VBG', 'IN']
--------------------------------------------------
Sentence : The weather is beautiful
POS Tags : ['DT', 'NN', 'VBZ', 'VBN']
--------------------------------------------------
Sentence : Artificial Intelligence changes the world
POS Tags : ['NNP', 'NNP', 'VBZ', 'DT', 'NN']
--------------------------------------------------
